# Expanding the Urdu OCR Dataset with UTRSet

This notebook pulls in real and synthetic Urdu text-line images from **UTRNet's UTRSet-Real and UTRSet-Synth** datasets (ICDAR'23, by Rahman, Ghosh & Arora — IIT Delhi) and folds a sample of them into your `labels.csv`.

Where this fits with your five categories:
- **UTRSet-Real** → real scanned printed Urdu text lines (Rekhta Foundation book/document scans). Good fit for your **newspaper** and **book** categories.
- **UTRSet-Synth** → computer-generated Urdu text images. Good fit for your **synthetic** category.
- **Signboard** and **handwriting** are NOT covered by this dataset. There isn't a small, freely downloadable public dataset for Urdu scene-text (signboards) or handwriting that I could verify. Your best bet for those two stays what it's been: your own photos for signboards, and (if you want a shortcut) reaching out to CLE Pakistan for their handwriting corpus, which requires contacting them directly.

**License note:** UTRSet is released under CC BY-NC-SA 4.0 for academic/research use. If you use it, cite the UTRNet paper (citation is in their GitHub README) in your project report.

Run the cells in order. Step 3 (inspect) matters — the exact internal folder layout isn't something I can verify from here, so check the printed output before running Step 4.


## Step 1: Install gdown and download

`gdown` is the standard tool for pulling files off Google Drive from the command line.


In [1]:
!pip install gdown --quiet



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# UTRSet-Real and UTRSet-Synth, from https://github.com/abdur75648/UTRNet-High-Resolution-Urdu-Text-Recognition
!gdown --fuzzy "https://drive.google.com/file/d/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T/view?usp=sharing" -O UTRSet-Real.zip
!gdown --fuzzy "https://drive.google.com/file/d/18hN2Ab2XtjiRJigogOd2DUBKa0zV0qDL/view?usp=sharing" -O UTRSet-Synth.zip


usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --fuzzy
usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --fuzzy


## Step 2: Extract


In [3]:
!mkdir -p UTRSet-Real UTRSet-Synth
!unzip -q -o UTRSet-Real.zip -d UTRSet-Real
!unzip -q -o UTRSet-Synth.zip -d UTRSet-Synth
print("Extraction done")


unzip:  cannot find or open UTRSet-Real.zip, UTRSet-Real.zip.zip or UTRSet-Real.zip.ZIP.
unzip:  cannot find or open UTRSet-Synth.zip, UTRSet-Synth.zip.zip or UTRSet-Synth.zip.ZIP.
Extraction done


## Step 3: Inspect the folder structure (do this before Step 4)

Google Drive zips don't always unpack into a flat, predictable layout. This cell prints the folder tree and looks for anything that could be a ground-truth label file (usually a `.txt` file mapping image paths to text). Read the output before touching Step 4.


In [4]:
import os

def print_tree(start_path, max_depth=3):
    for root, dirs, files in os.walk(start_path):
        depth = root.replace(start_path, "").count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... and {len(files) - 5} more files")

print("=== UTRSet-Real ===")
print_tree("UTRSet-Real")
print()
print("=== UTRSet-Synth ===")
print_tree("UTRSet-Synth")


=== UTRSet-Real ===
UTRSet-Real/

=== UTRSet-Synth ===
UTRSet-Synth/


In [5]:
import glob

# Look for likely label/ground-truth files
for folder in ["UTRSet-Real", "UTRSet-Synth"]:
    print(f"--- Possible label files in {folder} ---")
    candidates = glob.glob(f"{folder}/**/*.txt", recursive=True) + glob.glob(f"{folder}/**/*.csv", recursive=True)
    for c in candidates[:15]:
        print(" ", c)
    if candidates:
        print(f"\nFirst 3 lines of {candidates[0]}:")
        with open(candidates[0], "r", encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f):
                if i >= 3:
                    break
                print(" ", line.strip())
    print()


--- Possible label files in UTRSet-Real ---

--- Possible label files in UTRSet-Synth ---



## Step 4: Load labels and sample images

**Before running this**, update `REAL_GT_PATH` and `SYNTH_GT_PATH` below to match whatever label file Step 3 actually found. The code assumes each line looks like `image_path<TAB>text` (the convention UTRNet uses elsewhere in their repo) and falls back to splitting on the first space if there's no tab. If the real format is different (e.g. a CSV with headers), tell me what Step 3 printed and I'll adjust this cell.


In [6]:
import os
import glob

def find_label_files(folder):
    """Search for likely label/ground-truth files inside a folder."""
    candidates = glob.glob(f"{folder}/**/*.txt", recursive=True) + \
                 glob.glob(f"{folder}/**/*.csv", recursive=True) + \
                 glob.glob(f"{folder}/**/*.tsv", recursive=True)
    return candidates


def load_gt(gt_path, base_dir):
    rows = []
    with open(gt_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            if "\t" in line:
                img_rel, text = line.split("\t", 1)
            elif "," in line:
                img_rel, text = line.split(",", 1)
            else:
                img_rel, text = line.split(" ", 1)
            rows.append((os.path.join(base_dir, img_rel.strip()), text.strip()))
    return rows


def load_labels_auto(folder):
    candidates = find_label_files(folder)
    if not candidates:
        print(f"No .txt/.csv/.tsv label file found anywhere under '{folder}'.")
        print("Folder contents:")
        for root, dirs, files in os.walk(folder):
            for f in files[:10]:
                print(" ", os.path.join(root, f))
        return []

    print(f"Found {len(candidates)} candidate label file(s) under '{folder}':")
    for c in candidates:
        print(" ", c)

    # Try each candidate, use the first one that actually parses into rows
    for c in candidates:
        try:
            rows = load_gt(c, folder)
            if rows:
                print(f"\nUsing '{c}' — loaded {len(rows)} rows. First row: {rows[0]}")
                return rows
        except Exception as e:
            print(f"Could not parse '{c}': {e}")

    print("None of the candidate files parsed into usable rows.")
    return []


real_rows = load_labels_auto("UTRSet-Real")
synth_rows = load_labels_auto("UTRSet-Synth")

print(f"\nUTRSet-Real: {len(real_rows)} labeled images found")
print(f"UTRSet-Synth: {len(synth_rows)} labeled images found")

No .txt/.csv/.tsv label file found anywhere under 'UTRSet-Real'.
Folder contents:
No .txt/.csv/.tsv label file found anywhere under 'UTRSet-Synth'.
Folder contents:

UTRSet-Real: 0 labeled images found
UTRSet-Synth: 0 labeled images found


## Step 5: Sample, copy, and add to labels.csv

This picks a random sample from each source, copies the image files into your project's data folder, and appends new rows to `labels.csv`. Adjust `N_REAL` and `N_SYNTH` to however many you need to hit 200+ total (check with the audit cell from the Week 3 notebook).


In [7]:
import random
import shutil
import pandas as pd

random.seed(42)

# Adjust these to hit your 200-image target
N_REAL = 60
N_SYNTH = 40

# Point this at your existing Week 1 data folder (same as the Week 3 notebook)
DATA_DIR = "/workspaces/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/SI26-Week1/data"
LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
DEST_DIR = os.path.join(DATA_DIR, "raw", "external_utrset")
os.makedirs(DEST_DIR, exist_ok=True)

sample_real = random.sample(real_rows, min(N_REAL, len(real_rows)))
sample_synth = random.sample(synth_rows, min(N_SYNTH, len(synth_rows)))

new_rows = []

for i, (src_path, text) in enumerate(sample_real):
    fname = f"utrset_real_{i:04d}.jpg"
    dest_path = os.path.join(DEST_DIR, fname)
    shutil.copy(src_path, dest_path)
    new_rows.append({
        "image": os.path.join("raw", "external_utrset", fname),
        "text": text,
        "category": "newspaper"  # switch to "book" per-row if you can tell which is which
    })

for i, (src_path, text) in enumerate(sample_synth):
    fname = f"utrset_synth_{i:04d}.jpg"
    dest_path = os.path.join(DEST_DIR, fname)
    shutil.copy(src_path, dest_path)
    new_rows.append({
        "image": os.path.join("raw", "external_utrset", fname),
        "text": text,
        "category": "synthetic"
    })

new_df = pd.DataFrame(new_rows)
print(f"Prepared {len(new_df)} new rows")
new_df.head()


ModuleNotFoundError: No module named 'pandas'

In [ ]:
existing_df = pd.read_csv(LABELS_PATH)
combined_df = pd.concat([existing_df, new_df], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=["image"])
combined_df.to_csv(LABELS_PATH, index=False)

print(f"labels.csv had {len(existing_df)} rows, now has {len(combined_df)} rows")


## What's left

- Signboard and handwriting images still need to come from somewhere else — this dataset doesn't cover them.
- Run the Week 3 audit cell again to confirm your total and check for missing files.
- Don't forget the citation for UTRSet in your project report if you end up using these images in anything submitted or published.
